# Conexão com Banco de Dados Remoto

Este notebook demonstra como conectar Python a um servidor MySQL remoto e ler dados usando **pandas**.

Usaremos o banco público **Rfam** do Instituto Europeu de Bioinformática (EMBL-EBI):

| Parâmetro  | Valor                          |
|------------|--------------------------------|
| Host       | `mysql-rfam-public.ebi.ac.uk`  |
| Porta      | `4497`                         |
| Banco      | `Rfam`                         |
| Usuário    | `rfamro`                       |
| Senha      | *(nenhuma)*                    |

> **Rfam** é uma base de dados de famílias de RNA não-codificante. É pública, gratuita e somente leitura — perfeita para praticar consultas SQL sem precisar de cadastro ou senha.

## 1. Instalação das dependências

Precisamos de dois pacotes:
- `pymysql` — driver para conectar ao MySQL
- `pandas` — para exibir e manipular os dados em tabelas

In [7]:
%pip install pymysql pandas --quiet

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Python310\python.exe -m pip install --upgrade pip' command.


## 2. Importações

In [8]:
import pymysql
import pandas as pd

print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


## 3. Parâmetros de conexão

Centralizar os parâmetros em um dicionário facilita a manutenção e a reutilização.

In [9]:
params = {
    'host':     'mysql-rfam-public.ebi.ac.uk',
    'port':     4497,
    'database': 'Rfam',
    'user':     'rfamro',
    'password': '',
    'charset':  'utf8mb4',
}

## 4. Abrindo e testando a conexão

Usamos `try/except` para capturar erros de conexão e exibir uma mensagem clara.  
`conn` é inicializado como `None` antes do bloco para que as células seguintes possam verificar se a conexão está ativa.

In [10]:
conn = None

try:
    conn = pymysql.connect(**params)
    print('Conexão estabelecida!')
    print(f'Versão do servidor: {conn.get_server_info()}')
except pymysql.MySQLError as erro:
    print(f'Erro ao conectar: {erro}')

Conexão estabelecida!
Versão do servidor: 8.0.32-24


## 5. Listando as tabelas disponíveis

In [11]:
sql = 'SHOW TABLES'

tabelas = pd.read_sql(sql, conn)
print(f'{len(tabelas)} tabelas encontradas:\n')
print(tabelas.to_string(index=False))

64 tabelas encontradas:

             Tables_in_Rfam
            _annotated_file
               _family_file
               _genome_data
                      _lock
                   _overlap
        _overlap_membership
              _post_process
         alignment_and_tree
                     author
                       clan
         clan_database_link
  clan_literature_reference
            clan_membership
              database_link
                 db_version
                  dead_clan
                dead_family
              ensembl_names
                     family
              family_author
family_literature_reference
                family_long
                family_ncbi
                   features
                full_region
                     genome
                genome_temp
                     genseq
                genseq_temp
             html_alignment
                   keywords
       literature_reference
          matches_and_fasta
                      m

C:\Users\camila.kaufman\AppData\Local\Temp\ipykernel_11224\1646006481.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tabelas = pd.read_sql(sql, conn)


## 6. Lendo dados da tabela `family`

A tabela `family` contém o catálogo de famílias de RNA: nome, descrição e tipo.

In [25]:
sql = """
    SELECT *
    FROM   family_author
    LIMIT  20
"""

#"SELECT *" significar selecionar tudo
#rfam_acc, rfam_id, description, type, num_seed

df = pd.read_sql(sql, conn)
df

C:\Users\camila.kaufman\AppData\Local\Temp\ipykernel_11224\2383753866.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,rfam_acc,author_id,desc_order
0,RF01571,44,1
1,RF02588,2,1
2,RF02587,2,1
3,RF02586,2,1
4,RF02585,2,1
5,RF02549,2,1
6,RF02002,8,1
7,RF01741,56,1
8,RF00496,39,1
9,RF00469,39,1


## 7. Informações básicas sobre o DataFrame

In [26]:
print('Dimensões:', df.shape)
print()
df.info()

Dimensões: (20, 3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   rfam_acc    20 non-null     object
 1   author_id   20 non-null     int64 
 2   desc_order  20 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 608.0+ bytes


## 8. Filtrando por tipo

Quantas famílias existem de cada tipo? Vamos contar diretamente no SQL.

In [30]:
sql = """
    SELECT   type,
             COUNT(*)        AS total_familias,
             SUM(num_seed)   AS total_sequencias_seed
    FROM     family
    GROUP BY type
    ORDER BY total_sequencias_seed DESC
"""
#DESC - descendente
#ASC - acendente

df_tipos = pd.read_sql(sql, conn)
df_tipos

C:\Users\camila.kaufman\AppData\Local\Temp\ipykernel_11224\3911095272.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tipos = pd.read_sql(sql, conn)


,type,total_familias,total_sequencias_seed
0,Cis-reg;,351,25295.0
1,Gene; sRNA;,765,20937.0
2,Gene; miRNA;,1598,13938.0
3,Cis-reg; leader;,57,9271.0
4,Cis-reg; riboswitch;,52,7548.0
5,Intron;,17,6986.0
6,Gene; snRNA; snoRNA; CD-box;,484,4916.0
7,Gene; ribozyme;,35,3600.0
8,Gene;,73,3564.0
9,Gene; lncRNA;,218,3241.0


## 9. Busca com filtro de texto

Exemplo de `WHERE` com `LIKE`: buscando famílias cujo nome contém `'rRNA'`.

In [31]:
sql = """
    SELECT rfam_acc, rfam_id, description, num_seed
    FROM   family
    WHERE  rfam_id LIKE '%rRNA%'
    ORDER BY num_seed DESC
"""

df_rrna = pd.read_sql(sql, conn)
df_rrna

C:\Users\camila.kaufman\AppData\Local\Temp\ipykernel_11224\402458730.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rrna = pd.read_sql(sql, conn)


,rfam_acc,rfam_id,description,num_seed
0,RF00001,5S_rRNA,5S ribosomal RNA,712
1,RF04326,PV_xrRNA_class3,Exoribonuclease-resistant RNA (xrRNA) plant-in...,354
2,RF02541,LSU_rRNA_bacteria,Bacterial large subunit ribosomal RNA,102
3,RF00177,SSU_rRNA_bacteria,Bacterial small subunit ribosomal RNA,99
4,RF02540,LSU_rRNA_archaea,Archaeal large subunit ribosomal RNA,91
5,RF01960,SSU_rRNA_eukarya,Eukaryotic small subunit ribosomal RNA,90
6,RF02543,LSU_rRNA_eukarya,Eukaryotic large subunit ribosomal RNA,88
7,RF01959,SSU_rRNA_archaea,Archaeal small subunit ribosomal RNA,86
8,RF00002,5_8S_rRNA,5.8S ribosomal RNA,61
9,RF03547,Flavi_xrRNA,General Flavivirus exoribonuclease-resistant R...,48


## 10. Encerrando a conexão

Sempre feche a conexão ao terminar para liberar recursos no servidor remoto.

In [32]:
conn.close()
print('Conexão encerrada.')

Conexão encerrada.


---
## Resumo do fluxo

```
pymysql.connect(...)          # 1. abre a conexão
    │
    ├── pd.read_sql(sql, conn) # 2. executa SQL e retorna DataFrame
    │
    └── conn.close()           # 3. encerra a conexão
```

### Outros servidores públicos sem senha

| Servidor                         | Host                              | Porta | Usuário     |
|----------------------------------|-----------------------------------|-------|-------------|
| **Rfam** (RNA)                   | `mysql-rfam-public.ebi.ac.uk`     | 4497  | `rfamro`    |
| **Ensembl** (genômica)           | `ensembldb.ensembl.org`           | 3306  | `anonymous` |
| **Ensembl Genomes** (outros orgs)| `mysql-eg-publicsql.ebi.ac.uk`    | 4157  | `anonymous` |

---
*Fonte: [Rfam Public MySQL Database](https://docs.rfam.org/en/latest/database.html) · [Ensembl Public MySQL](http://www.ensembl.org/info/data/mysql.html)*

In [43]:
params = {
    'host':     'ensembldb.ensembl.org',
    'port':     3306,
    'database': '',
    'user':     'anonymous',
    'password': '',
    'charset':  'utf8mb4',
}

In [44]:
conn = None

try:
    conn = pymysql.connect(**params)
    print('Conexão estabelecida!')
    print(f'Versão do servidor: {conn.get_server_info()}')
except pymysql.MySQLError as erro:
    print(f'Erro ao conectar: {erro}')

Erro ao conectar: (1045, "Access denied for user 'camila.kaufman'@'189.20.43.84' (using password: NO)")


In [40]:
sql = 'SHOW DATABASES'

databases = pd.read_sql(sql, conn)
print(f'{len(databases)} tabelas encontradas:\n')
print(databases.to_string(index=False))

AttributeError: 'NoneType' object has no attribute 'cursor'

In [ ]:
impot basedosdados as bd

query = ""